## 00 — What Are Vector Tiles?

In Module 06 we identified four pain points in our handbuilt system:
1. Slow startup (load + index all data upfront)
2. Verbose GeoJSON format
3. Whole file always resident in memory
4. No streaming — unused regions still loaded

**Vector tiles** are the data format that solves all four. This notebook explains what they are before we use the tool that generates them.

## The Core Idea — Pre-Sliced, Pre-Indexed Data

Instead of four large files that cover the whole world, a vector tile pyramid pre-slices the world into thousands of small tiles — one per `{zoom}/{x}/{y}` address — before the user ever opens the map.

```
Zoom 0: 1 tile  (whole world)
Zoom 1: 4 tiles (quadrants)
Zoom 2: 16 tiles
...
Zoom 14: 268 million tiles (most empty)
```

When the user views a map, only the tiles that are currently visible are fetched. A user in Paris at zoom 12 receives ~12 tiles covering roughly 5km × 5km each. Siberia is never touched.

## How Each Tile Maps to Our System

Every choice we made manually now happens automatically inside the tile generator:

| What we built | What the tile system does |
|---------------|---------------------------|
| 4 LOD files at fixed epsilons | Per-zoom simplification baked into each tile |
| Grid index bucketing features into cells | Each tile IS a cell — features are pre-bucketed by definition |
| Viewport bbox culling | Each tile covers a fixed bbox — fetching only nearby tiles IS the cull |
| Zoom decision function | The tile URL scheme `/{z}/{x}/{y}` carries the zoom level |
| GeoJSON text format | MVT binary encoding — coordinates as integers, ~5× smaller |
| Whole file loaded at startup | Each tile fetched on demand, ~50–200 KB each |

## The Tile Coordinate System

Tiles use `(z, x, y)` addressing. At zoom `z`, the world is divided into a `2^z × 2^z` grid.

Given a longitude/latitude, we can compute its tile address:

In [ ]:
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y

# Paris
for zoom in [2, 5, 8, 12]:
    z, x, y = lon_lat_to_tile(2.35, 48.86, zoom)
    print(f"  zoom {z:>2}  tile ({z}/{x}/{y})")

## The MVT Binary Format

Mapbox Vector Tiles (MVT) store geometry as integers instead of floating-point text.

Each tile has a local coordinate space of 4096 × 4096 units. A coordinate like `(48.8566, 2.3522)` is projected into this space and stored as two small integers (e.g., `(2047, 1803)`).

This gives:
- **5–10× smaller files** vs. GeoJSON (integers compress better than decimal strings)
- **Faster parse** (no string-to-float conversion)
- **Lossy but controlled precision** (4096 units per tile at zoom 14 ≈ 2m resolution)

## The PMTiles Format

Traditionally, tile pyramids were stored in SQLite databases (`.mbtiles`) or as millions of individual files on a server.

**PMTiles** is a newer single-file format that stores the entire tile pyramid in one `.pmtiles` file, arranged so that spatially nearby tiles are stored close together on disk. A client can fetch just the tiles it needs using HTTP range requests — no tile server required, just a static file on any CDN.

For our purposes: `tippecanoe` can output either `.mbtiles` or `.pmtiles`.

## Exercise A

At zoom 12, the world is divided into `2^12 × 2^12 = 4096 × 4096 = ~16.7 million` tiles.

1. How many tiles cover Western Europe at zoom 12? (Approximate using the bounding box [-10, 35, 30, 60])
2. If each tile is 100 KB on average, how much data would the user need to download to view all of Western Europe at zoom 12?

Compare that to loading our `extra_fine` GeoJSON for the same region.

In [ ]:
from pathlib import Path

west_europe = (-10, 35, 30, 60)
zoom = 12

_, x_min, y_south = lon_lat_to_tile(west_europe[0], west_europe[1], zoom)
_, x_max, y_north = lon_lat_to_tile(west_europe[2], west_europe[3], zoom)

tiles_wide = x_max - x_min + 1
tiles_high = y_south - y_north + 1
total_tiles = tiles_wide * tiles_high

download_gb = total_tiles * 100_000 / 1_000_000_000  # 100 KB average tile
extra_fine_mb = Path('../../data/lod/railroads_extra_fine.geojson').stat().st_size / 1_000_000

print(f'x range: {x_min}..{x_max} ({tiles_wide} tiles)')
print(f'y range: {y_north}..{y_south} ({tiles_high} tiles)')
print(f'Total tiles covering Western Europe at z12: {total_tiles:,}')
print(f'At 100 KB per tile: {download_gb:.2f} GB of data')
print(f'Our extra_fine GeoJSON file: {extra_fine_mb:.2f} MB total worldwide')


## Exercise B

The tile coordinate formula uses the Web Mercator projection — the same projection used by Google Maps, OpenStreetMap, and virtually all web maps.

Web Mercator distorts areas significantly near the poles. Greenland appears roughly the same size as Africa on a Web Mercator map, even though Africa is ~14× larger.

Does this distortion affect the **accuracy** of our railroad visualization? Explain why or why not in 3–4 sentences.

In [ ]:
# Web Mercator distortion changes the apparent size and shape of regions,
# but it does not make the railroad overlay inaccurate as long as the tiles,
# basemap, and client all use the same projection. The lines still land in the
# correct visual position relative to the map. What distortion does affect is
# visual interpretation of area and distance, especially toward the poles.


## Check Your Understanding

The tile grid at zoom 14 has ~268 million possible tile addresses. Most tiles — over oceans, deserts, and polar regions — contain no data.

Both `.mbtiles` (SQLite) and `.pmtiles` (single file) only store non-empty tiles. Why is this critical, and how does it relate to the `scalerank` filtering decision we made in our LOD pipeline?

Sparse storage is critical because storing every possible tile address would waste enormous space on empty ocean, desert, and polar tiles. It solves the same broad problem as our `scalerank` coarse filter — avoiding useless data at low detail — but it does so spatially by omitting empty regions instead of using a hand-written global importance cutoff.


## Next

In [01 — Using Tippecanoe](./01-Using_Tippecanoe.ipynb), we run `tippecanoe` on the raw railroad GeoJSON and map each of its flags to decisions we already made.